# Benchmark on grouping v2 folds: YOLOv8m (3 seeds) + official split seeds 43/44

Completes the YOLOv8m row of the benchmark (docs/BENCHMARK_V2_PLAN.md). Same fixed protocol as notebooks 10-12: 100 epochs, no early stopping, `last.pt` (EMA), batch 16, imgsz 640, Ultralytics 8.4.215 defaults.
* 15 runs on the v2 folds (`data_splits_v2/clean5`, seeds 42/43/44 x folds 0-4) + 2 runs on the official split (seeds 43, 44; seed 42 exists from notebook 08) = 17 runs, about 6.5-7 h on T4 x2 (one account).
* `--save-val-preds`: each run also keeps the predictions Ultralytics scored on the test set and checks the shared offline evaluator against Ultralytics' own number (column "parity |diff|", must be about 0). This is what lets RF-DETR (notebook 15) be scored on exactly the same scale.

Kaggle settings: GPU T4 x2, Internet ON, HBFMID dataset attached. Optional first: `SMOKE = True` (2 epochs, 1 run, about 5 min).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
# faster-coco-eval is pre-installed so Ultralytics' save_json step never auto-installs it in the middle of a run
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215", "faster-coco-eval>=1.6.7"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert "--save-val-preds" in open(f"{REPO}/scripts/training/run_baseline.py").read(), "repo is behind: push the --save-val-preds patch first"
from ultralytics import YOLO, RTDETR

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
MODEL = "yolov8m.pt"
YOLO(MODEL); YOLO("yolo11n.pt")          # download once here (yolo11n.pt is used by the AMP self-check) so the two GPU workers do not race
V2 = f"{REPO}/data_splits_v2"; V1 = f"{REPO}/data_splits"          # official split lists live in data_splits/official
assert os.path.isdir(f"{V2}/clean5") and os.path.isdir(f"{V1}/official")
CLEAN_SEEDS, OFFICIAL_SEEDS, FOLDS = [42, 43, 44], [43, 44], [0, 1, 2, 3, 4]
if SMOKE:
    EPOCHS, OUT, ZIP = 2, "/kaggle/working/results/v2m_smoke", "v2m_smoke_results"; CLEAN_SEEDS, OFFICIAL_SEEDS, FOLDS = [42], [], [0]
else:
    EPOCHS, OUT, ZIP = 100, "/kaggle/working/results/v2m", "v2m_results"

def make_cmd(mode, fold, seed, splits):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", mode, "--fold", str(fold), "--seed", str(seed),
                "--model", MODEL, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", splits, "--out", OUT, "--save-val-preds"]
    return f
TASKS = [(f"official_s{s}_yolov8m", make_cmd("official", 0, s, V1)) for s in OFFICIAL_SEEDS]        # longest first
TASKS += [(f"v2mclean_f{f}_s{s}_yolov8m", make_cmd("clean", f, s, V2)) for s in CLEAN_SEEDS for f in FOLDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn = q.get_nowait()
        except queue.Empty:
            return
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import glob, json, shutil
print(f"{'run':40s} {'mAP50-95':>9s} {'mAP50':>7s} {'train_min':>9s} {'parity |diff|':>13s}  flag")
for p in sorted(glob.glob(f"{OUT}/*/metrics.json")):
    r = json.load(open(p)); m = r["test"]["map5095"]; par = r.get("test_offline_parity", {})
    d = par.get("abs_diff_map5095"); flag = "<-- collapsed? check the log" if m < 0.05 else ""
    if d is None:
        flag += " parity: " + str(par.get("error", "missing"))
    elif d > 1e-3:
        flag += " <-- parity off, tell Claude"
    print(f"{r['tag']:40s} {m:9.4f} {r['test']['map50']:7.4f} {r['train_seconds']/60:9.1f} {('%.2e' % d) if d is not None else 'n/a':>13s}  {flag}")
shutil.make_archive(f"/kaggle/working/{ZIP}", "zip", OUT)
print("\nzip:", f"/kaggle/working/{ZIP}.zip"); print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `v2m_results.zip`, unzip once into `results/` (any folder name; Claude renames it to `results/v2m`), then paste the printed table to Claude.